# 3.2 从一个简单的Vector 算子实现开始

## 概述

本章我们先实现一个简单的 ReLU 算子，其计算逻辑如下：负数变为零，非负数保留。我们用熟悉的三段式模板完成 ReLU，并通过典型用例开启后续学习。

## 学习目标

- 实现 ReLU 算子，完成读入、ReLU 计算与写回。
- 使用 shape 为 `(64,)`、`(1024,)`、`(8192,)`、`(65536,)` 的用例，验证 ReLU 的计算精度与运行边界。
- 练习实现一个 Exp 算子。

[本章导航](README.md) · [上一节](03.01_intro.ipynb) · [下一节](03.03_tiling.ipynb)

<p><small>运行前完成第 1.4 节的环境准备，并使用本章 README 指定的已验证运行入口启动新内核。每页独立执行，工作目录为本章根目录；内核本身不初始化 NPU。</small></p>

## 1. 功能实现与验证

ReLU 定义为 $y_i=\max(x_i,0)$。本节预设 `N=64、1024、8192、65536` 四个用例。先以 `N=64` 展示代码，令一份逻辑任务一次处理完整输入，沿用“读入—计算—写回”的三段式，再用这四种大小验证实现。

<img alt="ReLU 实现说明" src="images/03.02_relu.svg" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

三段式中，A 与 C 是 GM 中的完整输入和输出，a 与 c 是 UB 中的内存。读入保持数值不变；计算将负数改为零，保留非负数；写回把 c 保存到 C。图中以 shape=(64,) 为例，只展开少量示意数值，省略号表示其余元素。
计算部分先用 `vld` 从 UB 载入一组 64 个 fp32 元素，`vmax` 与零寄存器逐项比较，再用 `vsts` 写回 UB。`pset(32)` 的 32 表示元素位宽。GM 与 UB 之间仍使用 `T.copy`。

```python
import tilelang
import tilelang.language as T

N = 64


@T.prim_func
def relu(A: T.Tensor((N,), "float32"), C: T.Tensor((N,), "float32")):
    with T.Kernel(1) as bx:
        a = T.alloc_shared((N,), "float32")
        c = T.alloc_shared((N,), "float32")
        T.copy(A, a)  # 读入

        with T.SimdVF():
            mask = T.simd.pset(32)
            zero = T.simd.vdup(0.0, "float32", mask)
            for group in T.serial(N // 64):
                ra = T.simd.vld(a[group * 64])
                rc = T.simd.vmax(ra, zero, mask)
                T.simd.vsts(c[group * 64], rc, mask)  # ReLU 计算

        T.copy(c, C)  # 写回


kernel = tilelang.compile(relu, out_idx=-1)
```

`A` 与 `C` 的 shape 都是 `(N,)`；`a` 存输入，`c` 存结果。`T.simd.vmax(ra, zero, mask)` 对寄存器内各项取输入与零的较大值。读入用 `A → a`，写回用 `c → C`，输入与结果工作区不能混淆。

### 为什么按 64 个元素划分？

当前示例使用的向量寄存器宽度为 256 字节，fp32 每个元素占 4 字节，因此一组寄存器容纳 `256 / 4 = 64` 个元素（64 个 lane，即寄存器中的元素位置）。这里的 64 来自寄存器宽度与数据类型，不是核数，也不是 `pset(32)` 中的 32。

`for group in T.serial(N // 64)` 依次处理这些寄存器组，`group * 64` 是每组在 UB 中的起始元素下标。例如 `N=1024` 时共有 16 组，第 0 组处理下标 0～63，第 1 组处理 64～127，直到第 15 组处理 960～1023。每轮先用 `vld` 将一组输入从 a 读入寄存器，再用 `vmax` 与 `vdup` 生成的全零寄存器逐项比较，最后用 `vsts` 将这一组结果写入 c。`pset(32)` 按 32 位元素生成默认全有效掩码，启用这 64 个位置。

这种分组是一次 SimdVF 内部的寄存器处理粒度；a、c 仍各分配 N 个元素，并没有因此缩小 UB 工作区。当前循环只遍历完整组，未实现余数部分的安全读写与尾部掩码；支持其他长度需要补上相应处理，不能只把循环次数向上取整。

完整可运行实现见 [relu_basic.py](src/relu_basic.py) 的 `make_relu(N)`；它将上面的 N 参数化，并在入口检查支持范围。完整校验见 [run_relu_basic.py](src/run_relu_basic.py)。

### 验证用例与结果

以 `torch.relu` 为参考，对每个完整输出逐元素比较。输入为连续、有限的 fp32 NPU Tensor，seed=20260911，ReLU 要求精确相等（rtol=0、atol=0）。每个 shape 都覆盖随机值、全零、正常数以及跨零的线性序列；入口限定绝对值不超过 2048。

按预设的 `(64,)`、`(1024,)`、`(8192,)`、`(65536,)` 四个用例进行验证。结果如下，前三行的最大误差取四类输入中的最大值：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">输入 shape</th>
<th style="text-align: left; vertical-align: top;">验证内容</th>
<th style="text-align: left; vertical-align: top;">最大绝对误差</th>
<th style="text-align: left; vertical-align: top;">结果</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>(64,)</code></td>
<td style="text-align: left; vertical-align: top;">四类输入，完整输出比较</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">精度通过</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>(1024,)</code></td>
<td style="text-align: left; vertical-align: top;">四类输入，完整输出比较</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">精度通过</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>(8192,)</code></td>
<td style="text-align: left; vertical-align: top;">四类输入，完整输出比较</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">精度通过</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>(65536,)</code></td>
<td style="text-align: left; vertical-align: top;">四类输入，完整输出比较</td>
<td style="text-align: left; vertical-align: top;">—（无输出）</td>
<td style="text-align: left; vertical-align: top;">执行报错</td>
</tr>
</tbody>
</table>

前三种 shape 共 12 个完整输出全部通过。最后一行不是数值精度不达标，而是执行时报错，所以没有输出结果。其原因在本节第2小节“UB 空间与分核”中解释。

四个用例中，前三个执行核函数并比较精度，65536 项用例在入口检查时被校验拦截。下面展示前三个用例的执行输出。65536 项用例可运行 `python src/controlled_error.py` 复现预期的入口校验拦截。

In [1]:
import subprocess, sys, re
from pathlib import Path
from src.course_paths import OUTPUT_DIR
(OUTPUT_DIR / "validation").mkdir(parents=True, exist_ok=True)
# 独立子进程使用当前内核的解释器；非零返回码会中止本页。
command = [sys.executable, 'src/run_relu_basic.py', 'relu']
try:
    result = subprocess.run(command, check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as error:
    print(error.stdout)
    print(error.stderr)
    raise
log_name = Path(command[1]).stem + ("_" + command[2] if len(command)>2 else "") + "_notebook.log"
Path(OUTPUT_DIR / "validation", log_name).write_text(result.stdout + result.stderr)
# 输出节选：省略编译日志与安装路径；完整运行日志写入临时输出目录。
print("\n".join(line for line in result.stdout.splitlines()
                if not re.match(r"^\d{4}-\d{2}-\d{2}", line)
                    and not line.startswith("Loading tilelang libs from dev root:")))

relu N=64 random: full-output PASS; shape=(64,); max_abs_error=0
relu N=64 zero: full-output PASS; shape=(64,); max_abs_error=0
relu N=64 constant: full-output PASS; shape=(64,); max_abs_error=0
relu N=64 ramp: full-output PASS; shape=(64,); max_abs_error=0
relu N=1024 random: full-output PASS; shape=(1024,); max_abs_error=0
relu N=1024 zero: full-output PASS; shape=(1024,); max_abs_error=0
relu N=1024 constant: full-output PASS; shape=(1024,); max_abs_error=0
relu N=1024 ramp: full-output PASS; shape=(1024,); max_abs_error=0
relu N=8192 random: full-output PASS; shape=(8192,); max_abs_error=0
relu N=8192 zero: full-output PASS; shape=(8192,); max_abs_error=0
relu N=8192 constant: full-output PASS; shape=(8192,); max_abs_error=0
relu N=8192 ramp: full-output PASS; shape=(8192,); max_abs_error=0
Verification passed!


## 2. UB 空间与分核

为什么同一个 ReLU，输入放大到 `(65536,)` 后便不能按当前写法继续执行？计算公式没有改变，变化的是一次需要放入 UB 的数据量。边界用例实际触发了以下入口检查：

```text
ValueError: Local workspaces exceed this backend's 248 KiB SimdVF budget
```

### UB 是什么，存放什么？

UB（Unified Buffer）是向量核使用的片上局部存储，用来保留当前计算所需的数据。GM（Global Memory）存放完整输入和输出；计算时先将数据从 GM 搬到 UB，处理后再写回 GM。UB 空间有限，不能因为 GM 中的 Tensor 变大，就假定片上也能容纳同样大的工作区。

在本例中，`T.alloc_shared((N,), "float32")` 申请对应的 UB 工作区。变量 `a` 和 `c` 分别保留输入和结果，是两个独立的一维 Tensor，不是同一块存储的两个名字。读入后 `a` 中有待计算的数据，计算写入 `c`，最后把 `c` 写回 `C`。计算这段代码的空间需求时，必须同时计算两份内存。

### 两份内存占多少空间？

fp32 每项占 4 字节。两份工作区合计为 `N × 4 + N × 4 = 8N` 字节；1 KiB=1024 字节。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">N</th>
<th style="text-align: left; vertical-align: top;">a 的空间</th>
<th style="text-align: left; vertical-align: top;">c 的空间</th>
<th style="text-align: left; vertical-align: top;">合计</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">64</td>
<td style="text-align: left; vertical-align: top;">0.25 KiB</td>
<td style="text-align: left; vertical-align: top;">0.25 KiB</td>
<td style="text-align: left; vertical-align: top;">0.5 KiB</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">1024</td>
<td style="text-align: left; vertical-align: top;">4 KiB</td>
<td style="text-align: left; vertical-align: top;">4 KiB</td>
<td style="text-align: left; vertical-align: top;">8 KiB</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">8192</td>
<td style="text-align: left; vertical-align: top;">32 KiB</td>
<td style="text-align: left; vertical-align: top;">32 KiB</td>
<td style="text-align: left; vertical-align: top;">64 KiB</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">65536</td>
<td style="text-align: left; vertical-align: top;">256 KiB</td>
<td style="text-align: left; vertical-align: top;">256 KiB</td>
<td style="text-align: left; vertical-align: top;">512 KiB</td>
</tr>
</tbody>
</table>

当前目标平台的 CANN 配置报告当前环境可分配的UB容量为 253952 字节，即 248 KiB；本例后端采用相同的容量预算。计算需要的 512 KiB 超过当前环境可分配的UB容量，用例 shape 为 `(65536,)` 时执行失败。

容量图如下，对比当前环境可分配的UB容量与计算需要使用的 UB 空间大小。

<img alt="机制图" src="images/03.02_capacity.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

左侧展示 N=1024 时的 ReLU 数据流：A 读入 a，计算结果存入 c，再写回 C。右侧 N=65536 时，a、c 各需 256 KiB，合计 512 KiB，超过当前环境可分配的UB容量（248 KiB）。

### 该怎么解决这个问题？

解决大输入问题，关键在于由于 UB 空间限制，整份输入不能一次塞进 UB。可以每次只搬入一部分数据，使用固定大小 UB 内存，完成一块后再处理下一块。这样，完整 Tensor 的大小与一次计算的工作区大小就可以分开考虑。

另外，当前 `T.Kernel(1)` 只描述一份逻辑任务；昇腾 NPU 有多个 vector 核（64/72），这样会导致大部分核并没有参与计算，导致资源的浪费。因此除了分块，我们通常还会把数据分配给不同核处理。**分块解决每次处理多大数据，分核解决这些任务交给哪些核。** 多个核各有独立的 UB 空间。

本节只提出这两个思路。下一节 3.3 将详细说明 tile 大小、核间任务分配及其索引关系。

## 小结

本节实现了 ReLU 算子功能，构造并验证四种 shape 用例；前三种精度通过，65536 执行失败，说明算子的实现除了正确的公式，还需要合理地进行存储空间分配。接下来先用 Exp 练习巩固三段式，再进入 tiling 分核。

## 课后练习：实现 Exp

保留读入和写回，只把计算替换为 `rc = T.simd.vexp(ra, mask)`。用 `torch.exp` 检查 shape 为 `(64,)`、`(1024,)`、`(8192,)` 的完整输出，覆盖随机、零、常数及跨零线性序列。

本练习使用有限 fp32 输入，入口限定绝对值不超过 10，比较容差固定为 rtol=1e-5、atol=1e-6。指数计算允许小的浮点误差，不因失败放宽容差；本节不承诺任意大幅值输入的指数仍有有限输出。

完整 Exp 实现（[relu_basic.py](src/relu_basic.py)） 中的 `make_exp(N)` 与 ReLU 只改变计算表达；[答案入口](answer/03.02_exp.py) 验证 Exp 的完整输出。

<details>
<summary>展开练习解析</summary>

保留读入、写回以及当前版本要求的 SimdVF 骨架，将计算改为 `rc = T.simd.vexp(ra, mask)`。完整实现见 src/relu_basic.py 的 make_exp；answer/03.02_exp.py 验证三种 shape 的四类输入，使用 torch.exp 参考和 rtol=1e-5、atol=1e-6。

</details>

In [2]:
import subprocess, sys, re
from pathlib import Path
from src.course_paths import OUTPUT_DIR
(OUTPUT_DIR / "validation").mkdir(parents=True, exist_ok=True)
# 独立子进程使用当前内核的解释器；非零返回码会中止本页。
command = [sys.executable, 'answer/03.02_exp.py']
try:
    result = subprocess.run(command, check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as error:
    print(error.stdout)
    print(error.stderr)
    raise
log_name = Path(command[1]).stem + ("_" + command[2] if len(command)>2 else "") + "_notebook.log"
Path(OUTPUT_DIR / "validation", log_name).write_text(result.stdout + result.stderr)
# 输出节选：省略编译日志与安装路径；完整运行日志写入临时输出目录。
print("\n".join(line for line in result.stdout.splitlines()
                if not re.match(r"^\d{4}-\d{2}-\d{2}", line)
                    and not line.startswith("Loading tilelang libs from dev root:")))

exp N=64 random: full-output PASS; shape=(64,); max_abs_error=0
exp N=64 zero: full-output PASS; shape=(64,); max_abs_error=0
exp N=64 constant: full-output PASS; shape=(64,); max_abs_error=0
exp N=64 ramp: full-output PASS; shape=(64,); max_abs_error=0
exp N=1024 random: full-output PASS; shape=(1024,); max_abs_error=0
exp N=1024 zero: full-output PASS; shape=(1024,); max_abs_error=0
exp N=1024 constant: full-output PASS; shape=(1024,); max_abs_error=0
exp N=1024 ramp: full-output PASS; shape=(1024,); max_abs_error=0
exp N=8192 random: full-output PASS; shape=(8192,); max_abs_error=0
exp N=8192 zero: full-output PASS; shape=(8192,); max_abs_error=0
exp N=8192 constant: full-output PASS; shape=(8192,); max_abs_error=0
exp N=8192 ramp: full-output PASS; shape=(8192,); max_abs_error=0
Verification passed!


[本章导航](README.md) · [上一节](03.01_intro.ipynb) · [下一节](03.03_tiling.ipynb)